In [44]:
import pandas as pd

In [45]:
df_features = pd.read_feather('df_features.fth')
df_labels = pd.read_feather('df_labels.fth')

In [47]:
df_labels
df_text_a = df_features[['index','headline']].rename(columns={'index':'headlinea','headline':'texta'})
df_text_b = df_features[['index','headline']].rename(columns={'index':'headlineb','headline':'textb'})

In [48]:
df_merged = df_labels.merge(df_text_a, on=['headlinea'],how='left').merge(df_text_b, on=['headlineb'],how='left')
df_merged['pairkey'] = df_merged.apply(lambda row: min(row['headlinea'],row['headlineb'])+max(row['headlinea'],row['headlineb']),axis=1)
df_merged['pairkey'].value_counts().head()

pairkey
201MID252MID        2
36358NYT37299NYT    2
30505NYT37010NYT    2
24875NYT30148NYT    2
126MID127MID        1
Name: count, dtype: int64

In [49]:
#There are some repeated pairs of events
df_merged_nodups= df_merged.drop_duplicates(subset=['pairkey'])


In [50]:
df_merged_nodups.coref.value_counts()

coref
0.0    28661
1.0     5733
Name: count, dtype: int64

In [51]:
#There might be identical headlines
# Remove also that case

In [53]:
df_merged_nodups.query('texta==textb')

,index,headlinea,headlineb,coref,split,texta,textb,pairkey
7466,3410,227MID,239MID,1.0,test,Chadian defence minister says two soldiers kil...,Chadian defence minister says two soldiers kil...,227MID239MID


In [66]:
df_merged_nodups_final= df_merged_nodups.query('texta!=textb')

In [67]:
#Turn coref label into integer
df_merged_nodups_final["coref"]= df_merged_nodups_final["coref"].astype(int)

In [69]:
#Leave relevant columns only
df_merged_nodups_final = df_merged_nodups_final.drop(["index","headlinea","headlineb","pairkey"],axis=1).reset_index()

In [71]:
df_merged_nodups_final.to_excel('validar.xlsx')

In [72]:
df_merged_nodups_final.to_feather("HoWMerge.fth")